# Aurora — avatar falante no Kaggle GPU

Este notebook prepara um primeiro teste de aproximadamente 8 segundos sem face swap. O personagem, corpo, roupa, cenário, pose e identidade permanecem os do vídeo-base. A pipeline Wav2Lip recebe o vídeo-base e um áudio e altera principalmente a região da boca para sincronizá-la com a fala.

O modo `audio` é o primeiro teste real. O modo `text` fica preparado como contrato: texto precisa ser convertido em áudio por um motor TTS autorizado antes de entrar no mesmo passo de lip-sync. Este notebook não usa voz genérica para fingir que é a voz do usuário.

**Importante:** o repositório oficial do Wav2Lip informa uso pessoal, acadêmico e de pesquisa; uso comercial exige autorização separada.

## Arquivos que devem ser enviados ao Kaggle

Crie um Dataset privado chamado `aurora-avatar-inputs` e envie cópias dos arquivos: `VID-20260826-WA0002.mp4` e `voice_sample_ministro_luiz_andre.wav`. Não envie a imagem-base, pois este fluxo não faz troca de rosto. Os originais locais permanecem fora do Kaggle.

Depois, anexe esse Dataset ao Notebook. O caminho esperado será `/kaggle/input/aurora-avatar-inputs/`. Se o Kaggle gerar outro nome de Dataset, altere apenas `INPUT_DIR` na célula de configuração.

In [ ]:
# Configuração — não alterar os arquivos enviados; todas as saídas vão para /kaggle/working.
from pathlib import Path
import subprocess, sys, shutil

INPUT_DIR = Path('/kaggle/input/aurora-avatar-inputs')
VIDEO_NAME = 'VID-20260826-WA0002.mp4'
AUDIO_NAME = 'voice_sample_ministro_luiz_andre.wav'
VIDEO_IN = INPUT_DIR / VIDEO_NAME
AUDIO_IN = INPUT_DIR / AUDIO_NAME
WORK = Path('/kaggle/working/aurora_test')
WORK.mkdir(parents=True, exist_ok=True)
VIDEO_COPY = WORK / 'video_base_copy.mp4'
AUDIO_COPY = WORK / 'audio_copy.wav'
AUDIO_8S = WORK / 'audio_8s.wav'
OUTPUT = WORK / 'aurora_wav2lip_test_8s.mp4'

assert VIDEO_IN.exists(), f'Vídeo não encontrado: {VIDEO_IN}'
assert AUDIO_IN.exists(), f'Áudio não encontrado: {AUDIO_IN}'
shutil.copy2(VIDEO_IN, VIDEO_COPY)
shutil.copy2(AUDIO_IN, AUDIO_COPY)
print('Entradas encontradas. As inferências usarão apenas cópias de trabalho.')

In [ ]:
# A execução deve parar se o Kaggle não tiver GPU CUDA.
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU CUDA não disponível. Ative GPU nas configurações do Kaggle; não execute em CPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## Instalação mínima — executar somente no Kaggle

Esta célula instala o código open source e dependências do Wav2Lip no ambiente temporário do Kaggle. Ela ainda precisa baixar os pesos oficiais antes da inferência. Não executar nesta sessão do Aurora.

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/Rudrabha/Wav2Lip.git
%cd /kaggle/working/Wav2Lip
!pip install -q -r requirements.txt
!mkdir -p face_detection/detection/sfd checkpoints
# Baixar aqui os dois pesos oficiais indicados no README do repositório:
# 1) s3fd.pth em face_detection/detection/sfd/s3fd.pth
# 2) wav2lip_gan.pth ou wav2lip.pth em checkpoints/
# Use os links oficiais do README; não substitua por pesos de terceiros.

In [ ]:
# Preparar cópias: áudio PCM mono 16 kHz e corte de 8 segundos.
!ffmpeg -y -i "{AUDIO_COPY}" -t 8 -ac 1 -ar 16000 -c:a pcm_s16le "{AUDIO_8S}"
!ffmpeg -y -i "{VIDEO_COPY}" -t 8 -c:v libx264 -c:a aac "{WORK / 'video_8s_copy.mp4'}"
print('Somente cópias foram convertidas/cortadas; os arquivos de entrada permanecem intactos.')

In [ ]:
# Inferência de áudio -> vídeo. O vídeo-base continua sendo a identidade do personagem.
VIDEO_8S = WORK / 'video_8s_copy.mp4'
CHECKPOINT = Path('/kaggle/working/Wav2Lip/checkpoints/wav2lip_gan.pth')
assert CHECKPOINT.exists(), 'Checkpoint não encontrado. Baixe apenas o peso oficial indicado no README.'
!python /kaggle/working/Wav2Lip/inference.py --checkpoint_path "{CHECKPOINT}" --face "{VIDEO_8S}" --audio "{AUDIO_8S}" --outfile "{OUTPUT}" --pads 0 20 0 0
print('Saída:', OUTPUT)

## Entrada por texto — preparada, mas não simulada

Para texto, o Aurora deve primeiro chamar um motor TTS autorizado e receber um WAV. O arquivo WAV entra exatamente na mesma função de lip-sync. Não foi incluído TTS genérico, porque isso não representaria a voz autorizada do usuário.

Fluxo futuro: `texto → TTS autorizado com voz consentida → WAV PCM → Wav2Lip → MP4`.

Fluxo já preparado: `áudio WAV do usuário → Wav2Lip → MP4`.

In [ ]:
def gerar_avatar_a_partir_de_audio(audio_wav, video_base=VIDEO_COPY, duracao=8):
    '''Contrato para áudio direto; executa apenas depois do ambiente GPU e pesos estarem confirmados.'''
    raise NotImplementedError('A célula de inferência acima é o primeiro teste autorizado; conecte o áudio WAV e execute no Kaggle.')

def gerar_avatar_a_partir_de_texto(texto):
    '''Contrato para texto; requer TTS autorizado e não usa voz genérica.'''
    raise NotImplementedError('Falta conectar um motor TTS autorizado para transformar texto em WAV.')

In [ ]:
# Validação pós-processamento. Executar somente após a inferência.
import json, subprocess
probe = subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration:stream=codec_name,codec_type,width,height,r_frame_rate','-of','json',str(OUTPUT)])
print(json.dumps(json.loads(probe), indent=2))
print('Inspecione visualmente: contorno da boca, dentes, olhos, piscadas, sobrancelhas, cabeça e continuidade entre quadros.')